<a href="https://colab.research.google.com/github/elviakiran-miranda-hue/BUS4118S26/blob/dev/GRP_4_AI_in_Marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# load dataset
df = pd.read_excel("/content/Marketing_Analytics_Dataset_by_Slidescope.xlsx")

# rename columns if needed
df = df.rename(columns={
    "Date": "date",
    "Marketing_Channel": "channel", # Corrected from "Platform"
    "Spend": "spend", # Corrected from "Cost"
    "Impressions": "impressions", # Added
    "Clicks": "clicks",           # Added
    "Conversions": "conversions"  # Added
})

# keep only needed columns
df = df[["date", "channel", "spend", "impressions", "clicks", "conversions"]]

# pick 3 channels
df = df[df["channel"].isin(["Search", "Social", "Email"])]

# limit to ~21 days
df = df.sort_values("date").head(63)  # 21 days * 3 channels

# save cleaned file
df.to_csv("cleaned_ad_data.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '/content/Marketing_Analytics_Dataset_by_Slidescope.xlsx'

In [ ]:
import pandas as pd
from typing import Dict, List, Tuple


class AdOptimizationAgent:
    def __init__(
        self,
        total_daily_budget: float = 300.0,
        min_channel_share: float = 0.15,
        max_daily_shift: float = 0.20,
        shift_amount: float = 0.10,
        zero_alloc_limit_days: int = 2,
        optimize_for: str = "conversions",
    ):
        self.total_daily_budget = total_daily_budget
        self.min_channel_share = min_channel_share
        self.max_daily_shift = max_daily_shift
        self.shift_amount = shift_amount
        self.zero_alloc_limit_days = zero_alloc_limit_days
        self.optimize_for = optimize_for

        self.logs: List[Dict] = []

    def load_data(self, csv_path: str) -> pd.DataFrame:
        df = pd.read_csv("/content/cleaned_ad_data.csv")
        required_cols = ["date", "channel", "spend", "impressions", "clicks", "conversions"]

        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values(["date", "channel"]).reset_index(drop=True)
        return df

    def calculate_metrics(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df["ctr"] = df.apply(
            lambda row: row["clicks"] / row["impressions"] if row["impressions"] > 0 else 0.0,
            axis=1,
        )
        df["cvr"] = df.apply(
            lambda row: row["conversions"] / row["clicks"] if row["clicks"] > 0 else 0.0,
            axis=1,
        )
        df["cpa"] = df.apply(
            lambda row: row["spend"] / row["conversions"] if row["conversions"] > 0 else float("inf"),
            axis=1,
        )

        return df

    def get_score(self, row: pd.Series) -> float:
        if self.optimize_for == "ctr":
            return row["ctr"]

        # Default: optimize for conversions, using CVR with a small CPA penalty
        if row["cpa"] == float("inf"):
            return 0.0
        return row["cvr"] * 0.8 + (1 / (row["cpa"] + 1)) * 0.2

    def initialize_budget(self, channels: List[str]) -> Dict[str, float]:
        equal_share = 1.0 / len(channels)
        return {channel: equal_share for channel in channels}

    def apply_guardrails(self, proposed: Dict[str, float], previous: Dict[str, float]) -> Dict[str, float]:
        adjusted = proposed.copy()

        # Cap per-day change for each channel
        for channel in adjusted:
            prev = previous[channel]
            lower = max(self.min_channel_share, prev - self.max_daily_shift)
            upper = min(1.0, prev + self.max_daily_shift)
            adjusted[channel] = min(max(adjusted[channel], lower), upper)

        # Ensure floor
        for channel in adjusted:
            adjusted[channel] = max(adjusted[channel], self.min_channel_share)

        # Normalize back to sum to 1
        total = sum(adjusted.values())
        adjusted = {k: v / total for k, v in adjusted.items()}

        # Recheck floor after normalization
        below_floor = [k for k, v in adjusted.items() if v < self.min_channel_share]
        if below_floor:
            adjusted = self._fix_minimum_floor(adjusted)

        return adjusted

    def _fix_minimum_floor(self, budget: Dict[str, float]) -> Dict[str, float]:
        channels = list(budget.keys())
        fixed = budget.copy()

        for ch in channels:
            if fixed[ch] < self.min_channel_share:
                fixed[ch] = self.min_channel_share

        remainder = 1.0 - sum(fixed.values())
        if remainder < 0:
            # Scale channels above the floor downward
            above_floor = [ch for ch in channels if fixed[ch] > self.min_channel_share]
            excess = sum(fixed[ch] - self.min_channel_share for ch in above_floor)

            if excess > 0:
                for ch in above_floor:
                    reducible = fixed[ch] - self.min_channel_share
                    reduction = (-remainder) * (reducible / excess)
                    fixed[ch] -= reduction

        total = sum(fixed.values())
        return {k: v / total for k, v in fixed.items()}

    def propose_next_budget(
        self,
        day_data: pd.DataFrame,
        previous_budget: Dict[str, float],
    ) -> Tuple[Dict[str, float], str]:
        day_data = day_data.copy()
        day_data["score"] = day_data.apply(self.get_score, axis=1)

        ranked = day_data.sort_values("score", ascending=False)
        best_channel = ranked.iloc[0]["channel"]
        worst_channel = ranked.iloc[-1]["channel"]

        proposed = previous_budget.copy()
        proposed[best_channel] += self.shift_amount
        proposed[worst_channel] -= self.shift_amount

        proposed = self.apply_guardrails(proposed, previous_budget)

        best_row = ranked.iloc[0]
        worst_row = ranked.iloc[-1]

        reason = (
            f"{best_channel} up due to strongest performance "
            f"(CTR={best_row['ctr']:.3f}, CVR={best_row['cvr']:.3f}, CPA={best_row['cpa']:.2f}); "
            f"{worst_channel} down due to weaker performance "
            f"(CTR={worst_row['ctr']:.3f}, CVR={worst_row['cvr']:.3f}, CPA={worst_row['cpa']:.2f})."
        )

        return proposed, reason

    def simulate_agent(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        df = self.calculate_metrics(df)

        channels = sorted(df["channel"].unique().tolist())
        dates = sorted(df["date"].unique())

        current_budget = self.initialize_budget(channels)

        decisions = []
        performance_rows = []

        for current_date in dates:
            day_data = df[df["date"] == current_date].copy()

            # Expected output using current budget and historical row efficiency
            day_total_conversions = 0.0
            day_total_clicks = 0.0
            day_total_spend = 0.0

            for _, row in day_data.iterrows():
                channel = row["channel"]
                allocated_spend = self.total_daily_budget * current_budget[channel]

                spend_ratio = allocated_spend / row["spend"] if row["spend"] > 0 else 0.0
                est_impressions = row["impressions"] * spend_ratio
                est_clicks = row["clicks"] * spend_ratio
                est_conversions = row["conversions"] * spend_ratio

                performance_rows.append(
                    {
                        "date": current_date,
                        "channel": channel,
                        "allocated_share": current_budget[channel],
                        "allocated_spend": allocated_spend,
                        "estimated_impressions": est_impressions,
                        "estimated_clicks": est_clicks,
                        "estimated_conversions": est_conversions,
                    }
                )

                day_total_spend += allocated_spend
                day_total_clicks += est_clicks
                day_total_conversions += est_conversions

            next_budget, reason = self.propose_next_budget(day_data, current_budget)

            decisions.append(
                {
                    "date": current_date,
                    "current_budget": current_budget.copy(),
                    "next_budget": next_budget.copy(),
                    "reason": reason,
                    "day_total_spend": day_total_spend,
                    "day_total_clicks": day_total_clicks,
                    "day_total_conversions": day_total_conversions,
                    "day_ctr": day_total_clicks / sum(day_data["impressions"]) if sum(day_data["impressions"]) > 0 else 0.0,
                    "day_cpa": day_total_spend / day_total_conversions if day_total_conversions > 0 else float("inf"),
                }
            )

            current_budget = next_budget

        decisions_df = pd.DataFrame(decisions)
        performance_df = pd.DataFrame(performance_rows)

        return decisions_df, performance_df

    def simulate_baseline(self, df: pd.DataFrame) -> pd.DataFrame:
        df = self.calculate_metrics(df)

        channels = sorted(df["channel"].unique().tolist())
        equal_budget = self.initialize_budget(channels)

        rows = []
        for current_date in sorted(df["date"].unique()):
            day_data = df[df["date"] == current_date].copy()

            day_total_conversions = 0.0
            day_total_clicks = 0.0
            day_total_spend = 0.0
            day_total_impressions = 0.0

            for _, row in day_data.iterrows():
                channel = row["channel"]
                allocated_spend = self.total_daily_budget * equal_budget[channel]
                spend_ratio = allocated_spend / row["spend"] if row["spend"] > 0 else 0.0

                day_total_impressions += row["impressions"] * spend_ratio
                day_total_clicks += row["clicks"] * spend_ratio
                day_total_conversions += row["conversions"] * spend_ratio
                day_total_spend += allocated_spend

            rows.append(
                {
                    "date": current_date,
                    "baseline_spend": day_total_spend,
                    "baseline_clicks": day_total_clicks,
                    "baseline_conversions": day_total_conversions,
                    "baseline_ctr": day_total_clicks / day_total_impressions if day_total_impressions > 0 else 0.0,
                    "baseline_cpa": day_total_spend / day_total_conversions if day_total_conversions > 0 else float("inf"),
                }
            )

        return pd.DataFrame(rows)

    def evaluate(self, decisions_df: pd.DataFrame, baseline_df: pd.DataFrame) -> Dict[str, float]:
        agent_total_conversions = decisions_df["day_total_conversions"].sum()
        agent_total_clicks = decisions_df["day_total_clicks"].sum()
        agent_total_spend = decisions_df["day_total_spend"].sum()

        baseline_total_conversions = baseline_df["baseline_conversions"].sum()
        baseline_total_clicks = baseline_df["baseline_clicks"].sum()
        baseline_total_spend = baseline_df["baseline_spend"].sum()

        return {
            "agent_total_conversions": round(agent_total_conversions, 2),
            "baseline_total_conversions": round(baseline_total_conversions, 2),
            "conversion_lift_pct": round(
                ((agent_total_conversions - baseline_total_conversions) / baseline_total_conversions) * 100, 2
            ) if baseline_total_conversions > 0 else 0.0,
            "agent_total_clicks": round(agent_total_clicks, 2),
            "baseline_total_clicks": round(baseline_total_clicks, 2),
            "agent_avg_cpa": round(agent_total_spend / agent_total_conversions, 2) if agent_total_conversions > 0 else float("inf"),
            "baseline_avg_cpa": round(baseline_total_spend / baseline_total_conversions, 2) if baseline_total_conversions > 0 else float("inf"),
        }


def format_budget(budget_dict: Dict[str, float]) -> str:
    return ", ".join([f"{k}: {v:.1%}" for k, v in budget_dict.items()])


def main():
    csv_path = "ad_optimization_mock_data.csv"  # change this to your file path if needed

    agent = AdOptimizationAgent(
        total_daily_budget=300.0,
        min_channel_share=0.15,
        max_daily_shift=0.20,
        shift_amount=0.10,
        optimize_for="conversions",
    )

    df = agent.load_data(csv_path)
    decisions_df, performance_df = agent.simulate_agent(df)
    baseline_df = agent.simulate_baseline(df)
    results = agent.evaluate(decisions_df, baseline_df)

    print("\n=== DAILY DECISION LOG ===")
    for _, row in decisions_df.iterrows():
        print(f"\nDate: {row['date'].date()}")
        print(f"Current Budget -> {format_budget(row['current_budget'])}")
        print(f"Next Budget    -> {format_budget(row['next_budget'])}")
        print(f"Reason         -> {row['reason']}")
        print(f"Conversions    -> {row['day_total_conversions']:.2f}")
        print(f"CPA            -> {row['day_cpa']:.2f}")

    print("\n=== EVALUATION SUMMARY ===")
    for key, value in results.items():
        print(f"{key}: {value}")

    decisions_df.to_csv("agent_decision_log.csv", index=False)
    performance_df.to_csv("agent_performance_output.csv", index=False)
    baseline_df.to_csv("baseline_results.csv", index=False)

    print("\nSaved files:")
    print("- agent_decision_log.csv")
    print("- agent_performance_output.csv")
    print("- baseline_results.csv")


if __name__ == "__main__":
    main()